# Lumis 21: re-check Claude's part after Phase 5 (skin gate and photo descriptions)

Phase 5 changed what Claude is asked. One call now does the skin check **and** describes the photo for the
explanation (prompt `explain-v2.1`). The skin question is word for word the one notebook 15 validated, but the
instructions around it changed, so the skin gate must be checked again **before** user testing.

**Rules fixed before running (2026-11-17), same as notebook 15 for the gate.** Keep the skin gate on only if, with the
Phase 5 prompt:
1. it catches **at least 80%** of non-skin images (DermNet's removed non-skin images plus `photo_check/not_skin/`);
2. it wrongly rejects **at most 2%** of SCIN validation phone photos of the five conditions;
3. at least **95%** of API calls return a usable answer on the first try (no retries, like the server).

If the gate fails, `claude_check.json` gets `"skin_gate": false` (the description stays on) and the backend is redeployed.

**Descriptions (decides whether the prompt needs another revision before user testing, not whether it stays on):**
4. the description is shown (passes the filter) for **at least 70%** of SCIN validation photos;
5. the median reading level of shown descriptions is **grade 8 or below** (Flesch-Kincaid, NFR9).

Reported only: why descriptions were filtered, the "fit" Claude gives for the photo's **true** condition versus the
other four (does Claude's view line up with the label?), Claude's time per call, and your own photos.

SCIN **test** photos are not used: they are kept for one-time tests. The code below is the backend's
`app/ml/claude_check.py` and `app/readability.py`, unchanged (the notebook checks the fingerprint).

Needs the Colab secret `ANTHROPIC_API_KEY`. **CPU runtime.** About 10 to 15 minutes; answers are cached in Drive, so a
rerun costs nothing and continues where it stopped.

In [ ]:
CLAUDE_MODEL = 'claude-haiku-5-5'
WORKERS = 2                          # calls at once; low, so rate limits do not count as failures
GATE_MIN_CAUGHT, GATE_MAX_REJECTED, MIN_API_OK = 0.80, 0.02, 0.95
MIN_SHOWN, MAX_MEDIAN_GRADE = 0.70, 8.0
CLAUDE_CHECK_SHA = '097d5955a4c58fe8'    # fingerprint of backend/app/ml/claude_check.py this notebook was built from
NAMES = {'acne': 'Acne', 'eczema': 'Eczema', 'psoriasis': 'Psoriasis', 'scabies': 'Scabies',
         'tinea': 'Fungal skin infection (tinea or ringworm)'}          # same names as backend conditions.json

In [ ]:
!pip install -q anthropic
from google.colab import drive, userdata
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'
API_KEY = userdata.get('ANTHROPIC_API_KEY')

The next two cells are the backend code, unchanged.

In [ ]:
%%writefile claude_check.py
"""Claude's part of a screening: a skin check and a description of the photo, in one call (Phase 5).

Claude (vision) gets the photo, never the person's details, and answers four structured questions:
  1. skin_visible: is skin clearly visible?             no -> "No skin detected." (the skin gate)
  2. photo_quality / photo_problem                      recorded only
  3. seen_1, seen_2: two short sentences describing what the skin looks like, without naming any condition
  4. fit: for each of the five conditions, whether the visible features are "typical", "partly" typical or
     "not_typical" of it
The app shows the description and the fit for the condition LUMIS chose, nothing else (app/wording.py, explanation()).

Rules (declared 2026-11-17, before building; build plan, Phase 5):
  - Claude explains Lumis's result and never overrides it. The condition and band come only from the model.
    On the SCIN test cases Claude asked directly was right 55.4% of the time, Lumis v3 69.4%.
  - Claude can only make the advice more careful: no skin -> no result (as before); "not_typical" for Lumis's
    condition -> the advice recommends a doctor and shows clinics. The band is never changed.
  - The call starts when the photo arrives and runs while the model runs. The server waits for it at most
    CLAUDE_WAIT_SECONDS after the upload; if it is late, fails or its text does not pass the filter below, the
    result is shown with the general explanation only. Every outcome is recorded with the screening.
  - Question 1 is word for word the version validated in notebook 15 (skin gate). The surrounding prompt changed,
    so the gate should be re-checked on the notebook 15 sets.

History: until 2026-11-17 a third question asked which of the model's top two conditions the photo looked more like,
and could lower the band. Notebook 15's pre-declared rule did not switch that part on, so it was removed.
"""
from __future__ import annotations

import base64
import io
import json
import re
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
from PIL import Image

DEFAULT_MODEL = "claude-haiku-5-5"
PROMPT_VERSION = "explain-v2.1"
# v1 -> v2 (2026-11-17, after 25 runs on Cloud Run): v1 asked for "one or two sentences" in one field; 10 of 25
# answers had three sentences and were filtered. v2 asks for exactly two sentences in two separate fields, so the
# shape of the answer enforces the limit. The filter itself is unchanged.
# v2 -> v2.1 (same day, 5 more runs): 3 of 5 answers still had an extra sentence inside a field. The prompt is the
# same; the app now shows the first sentence of each field. Every word Claude wrote is still checked for blocked words.
MAX_EDGE = 1024              # about 1,300 image tokens; plenty to judge a skin photo
FITS = ("typical", "partly", "not_typical")

SYSTEM = ("You look at skin photos for Lumis, a skin-condition screening aid used in Malaysia. You do not diagnose. "
          "Part of your answer is shown to the person who took the photo, so write it kindly, in short sentences "
          "and simple everyday English. Answer only with the requested JSON.")

PROMPT = """Look at this photo.
1. skin_visible: is human skin clearly visible and the main subject of the photo?
2. photo_quality: "good" if the skin area is in focus, close enough and lit well enough to judge its appearance; otherwise "poor", and give the main photo_problem.
3. seen_1 and seen_2: describe what the skin in the photo looks like in exactly two short sentences, one sentence in each field, each 20 words or fewer: color, shape, size, texture, edges and pattern, and the body area if you can tell. seen_1 starts with "In your photo,". Describe only what is visible. Do not name or suggest any condition, disease, infection or cause, and do not give advice. If no skin is visible, give empty strings.
4. fit: for each condition below, how well do the visible features match its usual appearance? Answer "typical", "partly" or "not_typical".
{conditions}"""


def build_schema(classes: list[str]) -> dict:
    return {
        "type": "object",
        "properties": {
            "skin_visible": {"type": "boolean"},
            "photo_quality": {"type": "string", "enum": ["good", "poor"]},
            "photo_problem": {"type": "string",
                              "enum": ["none", "blurry", "too_dark", "too_far", "obstructed", "not_skin", "other"]},
            "seen_1": {"type": "string"},
            "seen_2": {"type": "string"},
            "fit": {"type": "object", "properties": {c: {"type": "string", "enum": list(FITS)} for c in classes},
                    "required": list(classes), "additionalProperties": False},
        },
        "required": ["skin_visible", "photo_quality", "photo_problem", "seen_1", "seen_2", "fit"],
        "additionalProperties": False,
    }


# ---------------------------------------------------------------- the filter on Claude's description

# Words that would turn a description into a diagnosis or a guess at a cause. Lower case, matched as word starts.
BLOCKED_WORDS = (
    "acne", "pimple", "comedo", "blackhead", "whitehead", "eczema", "dermatitis", "psoria", "scabies", "mite",
    "burrow", "tinea", "ringworm", "fung", "yeast", "candida", "infect", "bacteri", "virus", "viral", "diagnos",
    "disease", "disorder", "condition", "syndrome", "cancer", "melanoma", "carcinoma", "tumor", "tumour", "wart",
    "allerg", "rosacea", "hive", "urticaria", "shingles", "herpes", "chickenpox", "measles", "lupus", "vitiligo",
    "treat", "cream", "ointment", "medicine", "doctor", "contagious",
)
MAX_SEEN_WORDS = 50
MAX_SEEN_SENTENCES = 2
_blocked = re.compile(r"\b(" + "|".join(BLOCKED_WORDS) + r")", re.IGNORECASE)


def _sentences(text: str) -> list[str]:
    return [p for p in re.split(r"(?<=[.!?])\s+", text.strip()) if p] or [""]


def check_seen(text: str) -> tuple[str | None, str | None]:
    """Claude's description -> (text to show, None) or (None, reason it is not shown)."""
    text = " ".join((text or "").split())
    if not text:
        return None, "empty"
    if len(text.split()) > MAX_SEEN_WORDS:
        return None, "too long"
    if len(_sentences(text)) > MAX_SEEN_SENTENCES:
        return None, "too many sentences"
    hit = _blocked.search(text)
    if hit:
        return None, f"blocked word: {hit.group(1).lower()}"
    if not text.lower().startswith("in your photo"):
        return None, "does not start with 'In your photo'"
    if text[-1] not in ".!":
        text += "."
    return text, None


# ---------------------------------------------------------------- the call

@dataclass(frozen=True)
class ClaudeVerdict:
    ok: bool                         # False = Claude's part was skipped (API error, refusal, bad output, late)
    skin_visible: bool | None = None
    photo_quality: str | None = None
    photo_problem: str | None = None
    seen: str | None = None          # description that passed the filter, or None
    seen_raw: str | None = None      # what Claude wrote (kept for the log and RQ3 evidence; never shown if filtered)
    filtered: str | None = None      # why the description is not shown
    fit: dict = field(default_factory=dict)
    error: str | None = None
    model: str | None = None
    prompt_version: str = PROMPT_VERSION
    ms: float | None = None          # how long the call took

    @property
    def outcome(self) -> str:
        """Short word for the logs and the Server-Timing header."""
        if not self.ok:
            return "late" if self.error == "late" else f"skipped ({self.error})"
        return f"filtered ({self.filtered})" if self.filtered else "ok"

    def to_dict(self) -> dict:
        return {**asdict(self), "outcome": self.outcome}


def prepare_image(rgb: np.ndarray, max_edge: int = MAX_EDGE) -> bytes:
    """Decoded RGB array (already turned upright) -> JPEG bytes, long edge at most max_edge."""
    im = Image.fromarray(np.ascontiguousarray(rgb))
    if max(im.size) > max_edge:
        im.thumbnail((max_edge, max_edge), Image.LANCZOS)
    b = io.BytesIO()
    im.save(b, "JPEG", quality=90)
    return b.getvalue()


def build_request(jpeg: bytes, names: dict[str, str], model: str) -> dict:
    """names: condition id -> display name, for all five classes (alphabetical, so the order hints nothing)."""
    classes = sorted(names)
    listing = "\n".join(f"   {c}: {names[c]}" for c in classes)
    return {
        "model": model,
        "max_tokens": 1024,            # thinking tokens would count toward this limit, so keep headroom
        "thinking": {"type": "disabled"},   # Haiku 5.5 thinks by default; off keeps the call fast (NFR1)
        "system": SYSTEM,
        "messages": [{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg",
                                         "data": base64.standard_b64encode(jpeg).decode()}},
            {"type": "text", "text": PROMPT.format(conditions=listing)},
        ]}],
        "extra_body": {"output_config": {"format": {"type": "json_schema", "schema": build_schema(classes)}}},
    }


def parse_response(response, classes: list[str], model: str) -> ClaudeVerdict:
    if getattr(response, "stop_reason", None) == "refusal":
        return ClaudeVerdict(ok=False, error="refusal", model=model)
    text = next((blk.text for blk in response.content if getattr(blk, "type", None) == "text"), None)
    try:
        d = json.loads(text)
        fit = {c: str(d["fit"][c]).lower() for c in classes}
        if any(v not in FITS for v in fit.values()):
            raise ValueError("unknown fit value")
        skin = bool(d["skin_visible"])
        parts = [p for p in (str(d.get("seen_1", "")).strip(), str(d.get("seen_2", "")).strip()) if p]
        raw = " ".join(p if p[-1] in ".!?" else p + "." for p in parts)
        # v2.1: only the first sentence of each field is shown (Claude sometimes writes two per field). The
        # blocked-word check still covers everything Claude wrote, so trimming can never let a diagnosis through.
        firsts = [_sentences(p)[0] for p in parts]
        shown = " ".join(p if p[-1] in ".!?" else p + "." for p in firsts)
        hit = _blocked.search(raw)
        if not skin:
            seen, why = None, "no skin"
        elif hit:
            seen, why = None, f"blocked word: {hit.group(1).lower()}"
        else:
            seen, why = check_seen(shown)
        return ClaudeVerdict(ok=True, skin_visible=skin, photo_quality=str(d["photo_quality"]).lower(),
                             photo_problem=str(d["photo_problem"]).lower(), seen=seen,
                             seen_raw=raw, filtered=why if skin else None, fit=fit, model=model)
    except (TypeError, ValueError, KeyError) as exc:
        stop = getattr(response, "stop_reason", None)
        return ClaudeVerdict(ok=False, error=f"unreadable output ({stop}): {type(exc).__name__}", model=model)


class ClaudeChecker:
    def __init__(self, api_key: str, model: str = DEFAULT_MODEL, timeout: float = 10.0, client=None,
                 skin_gate: bool = True):
        if client is None:
            import anthropic
            # no retries: the server only waits a few seconds for Claude anyway
            client = anthropic.Anthropic(api_key=api_key, timeout=timeout, max_retries=0)
        self.client, self.model, self.skin_gate = client, model, skin_gate

    def check(self, rgb: np.ndarray, names: dict[str, str]) -> ClaudeVerdict:
        started = time.perf_counter()
        try:
            response = self.client.messages.create(**build_request(prepare_image(rgb), names, self.model))
            verdict = parse_response(response, sorted(names), self.model)
        except Exception as exc:                       # network, rate limit, server error: skip Claude's part
            verdict = ClaudeVerdict(ok=False, error=f"api error: {type(exc).__name__}", model=self.model)
        ms = round((time.perf_counter() - started) * 1000, 1)
        return ClaudeVerdict(**{**asdict(verdict), "ms": ms})


def late(model: str, waited_ms: float) -> ClaudeVerdict:
    return ClaudeVerdict(ok=False, error="late", model=model, ms=round(waited_ms, 1))


def gate(verdict: ClaudeVerdict | None, skin_gate: bool = True) -> str:
    """'no_skin' if Claude says there is no skin (and the gate is on), else 'ok'. Never changes a band."""
    if verdict is not None and verdict.ok and skin_gate and verdict.skin_visible is False:
        return "no_skin"
    return "ok"


@dataclass(frozen=True)
class ClaudeCheckConfig:
    skin_gate: bool = False          # notebook 15, section 3
    explain: bool = False            # Phase 5: describe the photo for the explanation
    model: str = DEFAULT_MODEL

    @property
    def any(self) -> bool:
        return self.skin_gate or self.explain

    @classmethod
    def load(cls, path: str | Path) -> "ClaudeCheckConfig":
        p = Path(path)
        if not p.exists():
            return cls()
        c = json.loads(p.read_text())
        return cls(skin_gate=bool(c.get("skin_gate", False)), explain=bool(c.get("explain", False)),
                   model=c.get("model", DEFAULT_MODEL))

In [ ]:
%%writefile readability.py
"""Flesch-Kincaid grade level (NFR9: everything Lumis says should read at grade 8 or below).

Grade = 0.39 * (words / sentences) + 11.8 * (syllables / words) - 15.59. Syllables are counted with the usual
vowel-group rule (silent final e dropped, at least one per word). That is an estimate, accurate enough to keep
texts plainly worded, which is the point; the CP2 report should say an estimate was used."""
import re

_word = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def syllables(word: str) -> int:
    w = word.lower().strip("'")
    if len(w) <= 3:
        return 1
    w = re.sub(r"(?:[^laeiouy]es|[^laeiouy]ed|[^laeiouy]e)$", "", w)   # silent endings: makes, used, skin-e
    w = re.sub(r"^y", "", w)
    return max(1, len(re.findall(r"[aeiouy]{1,2}", w)))


def fk_grade(text: str) -> float:
    words = _word.findall(text)
    if not words:
        return 0.0
    sentences = max(1, len([s for s in re.split(r"[.!?]+", text) if _word.search(s)]))
    syl = sum(syllables(w) for w in words)
    return round(0.39 * len(words) / sentences + 11.8 * syl / len(words) - 15.59, 1)

In [ ]:
import os, json, glob, time, zipfile, hashlib, threading
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps
import importlib, claude_check, readability
importlib.reload(claude_check); importlib.reload(readability)
from claude_check import ClaudeChecker, ClaudeVerdict, PROMPT_VERSION, gate
from readability import fk_grade

assert hashlib.sha256(open('claude_check.py', encoding='utf-8').read().strip().encode()).hexdigest()[:16] == CLAUDE_CHECK_SHA, \
    'claude_check.py differs from the backend version this notebook was built for.'
print('Prompt', PROMPT_VERSION, '| model', CLAUDE_MODEL)

OUT = f'{BASE}/exports/claude_explain'
os.makedirs(OUT, exist_ok=True)
for name, folder in [('scin_images.zip', '/content/scin'), ('dermnet.zip', '/content/dermnet')]:
    if not os.path.exists(folder):
        with zipfile.ZipFile(f'{BASE}/data/{name}') as z:
            z.extractall('/content' if name == 'scin_images.zip' else folder)

split = pd.read_csv(f'{BASE}/data/scin_split_v1.csv', dtype={'case_id': str})
scin = split[(split.split == 'val') & split.eval_photo].reset_index(drop=True)
scin['file'] = '/content/' + scin.path
removed = pd.read_csv(f'{BASE}/exports/removed_nonskin_v2.csv')

def open_rgb(path):
    """Upright RGB pixels (the backend turns phone photos upright the same way). None if it cannot be read."""
    try:
        return np.asarray(ImageOps.exif_transpose(Image.open(path)).convert('RGB'))
    except Exception:
        print('Skipped, cannot be read (HEIC must be saved as JPEG):', os.path.basename(path))
        return None

nonskin = {'DermNet non-skin': ['/content/dermnet/' + p for p in removed.path],
           'Your not-skin photos': sorted(f for f in glob.glob(f'{BASE}/data/photo_check/not_skin/*') if os.path.isfile(f))}
print(f'SCIN validation: {len(scin)} photos', scin.lumis_class.value_counts().to_dict())
print({k: len(v) for k, v in nonskin.items()})

## 1. Ask Claude (cached)

In [ ]:
checker = ClaudeChecker(API_KEY, CLAUDE_MODEL)          # same settings as the server: 10 s timeout, no retries
CACHE = f'{OUT}/answers_{PROMPT_VERSION}_{CLAUDE_MODEL}.jsonl'
cache, ever_failed, lock = {}, set(), threading.Lock()
if os.path.exists(CACHE):
    for line in open(CACHE):
        r = json.loads(line)
        if r['verdict'].get('ok'):
            cache[r['file']] = r['verdict']
        else:
            ever_failed.add(r['file'])
print(len(cache), 'answers already cached,', len(ever_failed), 'earlier failures')

def ask(file):
    if file in cache:
        return
    rgb = open_rgb(file)
    if rgb is None:
        return
    v = checker.check(rgb, NAMES)
    with lock, open(CACHE, 'a') as f:
        if v.ok:
            cache[file] = v.to_dict()
        else:                                          # retried on a rerun, but still counted as a failed call
            ever_failed.add(file)
        f.write(json.dumps({'file': file, 'verdict': v.to_dict()}) + '\n')

def verdicts(files):
    todo = [f for f in files if f not in cache]
    t0 = time.time()
    with ThreadPoolExecutor(WORKERS) as pool:
        list(pool.map(ask, todo))
    if todo:
        print(f'{len(todo)} calls in {time.time() - t0:.0f} s')
    return {f: ClaudeVerdict(**{k: v for k, v in cache[f].items() if k != 'outcome'}) for f in files if f in cache}

all_files = list(scin.file) + [f for fs in nonskin.values() for f in fs]
for attempt in range(2):                               # second pass retries failures (still counted against rule 3)
    V = verdicts(all_files)
print(len(V), 'answers for', len(all_files), 'photos')

## 2. Skin gate: decision

In [ ]:
def no_skin(v):
    return gate(v) == 'no_skin'

caught_by = {}
for name, files in nonskin.items():
    got = [f for f in files if f in V]
    caught_by[name] = (sum(no_skin(V[f]) for f in got), len(files))     # unreadable or failed = not caught
    print(f'{name}: caught {caught_by[name][0]} of {caught_by[name][1]}')
caught = sum(c for c, _ in caught_by.values()) / max(sum(n for _, n in caught_by.values()), 1)
scin_v = [V[f] for f in scin.file if f in V]
wrongly_rejected = float(np.mean([no_skin(v) for v in scin_v]))
first_try = float(np.mean([f not in ever_failed for f in all_files]))
gate_checks = {'non-skin images caught': (caught, caught >= GATE_MIN_CAUGHT),
               'SCIN photos wrongly rejected': (wrongly_rejected, wrongly_rejected <= GATE_MAX_REJECTED),
               'API calls answered first try': (first_try, first_try >= MIN_API_OK)}
for k, (v, ok) in gate_checks.items():
    print(f'{k:32s} {v:.3f}  {"meets the rule" if ok else "FAILS the rule"}')
skin_gate = all(ok for _, ok in gate_checks.values())
print('\nSkin gate with the Phase 5 prompt:', 'KEEP ON' if skin_gate else 'SWITCH OFF')

old = f'{BASE}/exports/claude_check/validation_decision.json'
if os.path.exists(old):
    o = json.load(open(old)).get('skin_gate_checks', {})
    print('Notebook 15 (old prompt), for comparison:', {k: round(v, 3) for k, v in o.items()})

## 3. Descriptions on SCIN validation photos

In [ ]:
rows = []
for f, cls in zip(scin.file, scin.lumis_class):
    v = V.get(f)
    if v is None:
        continue
    rows.append({'file': os.path.basename(f), 'true': cls, 'skin': v.skin_visible, 'shown': v.seen is not None,
                 'filtered': v.filtered, 'grade': fk_grade(v.seen) if v.seen else np.nan,
                 'words': len(v.seen.split()) if v.seen else np.nan, 'fit_true': v.fit.get(cls),
                 'ms': v.ms, 'seen': v.seen, 'seen_raw': v.seen_raw})
d = pd.DataFrame(rows)
with_skin = d[d.skin == True]
shown = float(with_skin.shown.mean())
median_grade = float(d.grade.median())
desc_checks = {'description shown': (shown, shown >= MIN_SHOWN),
               'median reading grade': (median_grade, median_grade <= MAX_MEDIAN_GRADE)}
for k, (v, ok) in desc_checks.items():
    print(f'{k:22s} {v:.3f}  {"meets the rule" if ok else "FAILS the rule"}')
print('\nWhy descriptions were not shown:', d.filtered.dropna().str.replace(r':.*', '', regex=True).value_counts().to_dict())
print('Blocked words:', d.filtered.dropna().str.extract(r'blocked word: (\w+)')[0].value_counts().to_dict())
print('Shown by true condition:', with_skin.groupby('true').shown.mean().round(2).to_dict())
print(f'Words per shown description: median {d.words.median():.0f}, max {d.words.max():.0f}')
print(f'Claude time per call (Colab to Anthropic): median {d.ms.median() / 1000:.2f} s, '
      f'90th percentile {d.ms.quantile(0.9) / 1000:.2f} s')
display(d[d.shown].sample(min(8, int(d.shown.sum())), random_state=0)[['true', 'fit_true', 'grade', 'seen']])

**Does Claude's "fit" line up with the label?** For each photo, the fit Claude gave the photo's true condition, and
on average the fit it gave the four other conditions. If Claude's view were useless, both would look alike. Reported
only: Claude never changes Lumis's result.

In [ ]:
fit_rows = []
for f, cls in zip(scin.file, scin.lumis_class):
    v = V.get(f)
    if v is None or not v.ok or not v.fit:
        continue
    for c, fit in v.fit.items():
        fit_rows.append({'true': cls, 'condition asked': c, 'is_true': c == cls, 'fit': fit})
F = pd.DataFrame(fit_rows)
fit_table = (F.groupby('is_true').fit.value_counts(normalize=True).unstack(fill_value=0)
             .reindex(columns=['typical', 'partly', 'not_typical'], fill_value=0).round(3))
fit_table.index = ['other conditions', 'true condition']
display(fit_table)
display(F[F.is_true].groupby('true').fit.value_counts(normalize=True).unstack(fill_value=0).round(2))

## 4. Save

In [ ]:
record = {'prompt_version': PROMPT_VERSION, 'claude_model': CLAUDE_MODEL, 'claude_check_sha': CLAUDE_CHECK_SHA,
          'n_scin_val': int(len(scin)), 'n_answered': len(V),
          'skin_gate': skin_gate, 'skin_gate_checks': {k: round(float(v), 4) for k, (v, _) in gate_checks.items()},
          'caught_by_source': {k: list(v) for k, v in caught_by.items()},
          'description_checks': {k: round(float(v), 4) for k, (v, _) in desc_checks.items()},
          'description_ok': all(ok for _, ok in desc_checks.values()),
          'filtered_reasons': d.filtered.dropna().value_counts().to_dict(),
          'fit_true_vs_other': {i: {k: float(x) for k, x in r.items()} for i, r in fit_table.iterrows()},
          'claude_ms_median': float(d.ms.median()), 'claude_ms_p90': float(d.ms.quantile(0.9)),
          'created_at': time.strftime('%Y-%m-%d %H:%M')}
json.dump(record, open(f'{OUT}/recheck.json', 'w'), indent=1, default=float)
d.drop(columns=['seen_raw']).to_csv(f'{OUT}/descriptions_scin_val.csv', index=False)
print(json.dumps({k: record[k] for k in ['skin_gate', 'skin_gate_checks', 'description_ok', 'description_checks']}, indent=1))
if not skin_gate:
    print('\nThe skin gate FAILED with the new prompt: set "skin_gate": false in backend/models/claude_check.json and redeploy.')
if not record['description_ok']:
    print('\nThe descriptions need another prompt revision before user testing (see the reasons above).')
print('\nSaved', f'{OUT}/recheck.json', 'and descriptions_scin_val.csv')

## 5. Your own photos (reported only)

In [ ]:
own = []
for sub in ['healthy_skin', 'not_skin', 'real_conditions']:
    files = sorted(f for f in glob.glob(f'{BASE}/data/photo_check/{sub}/*') if os.path.isfile(f))
    for f, v in verdicts(files).items():
        own.append({'set': sub, 'file': os.path.basename(f), 'skin': v.skin_visible, 'outcome': v.outcome,
                    'seen': v.seen})
own = pd.DataFrame(own)
if len(own):
    display(own.groupby('set').outcome.value_counts().unstack(fill_value=0))
    display(own[own.set != 'not_skin'][['set', 'file', 'seen']])